# set up

In [8]:
import sys
if sys.platform == 'linux':
    sys.path.append("/home/qix/MultiNeuronGLM")
else:
    sys.path.append("D:/Github/MultiNeuronGLM")

import copy
import pandas as pd
import utility_functions as utils
import GLM
from DataLoader import Allen_dataset
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import numpy as np
sns.set_theme()

import torch
from torch.autograd import Variable
from torch.nn import functional as F

In [2]:
# Use self trained K-means results, which have a better likelihood
import pickle
with open('group_id_all_a_c/membership.pickle', 'rb') as handle:
    membership = pickle.load(handle)
with open('group_id_all_a_c/condition_ids.pickle', 'rb') as handle:
    condition_ids = pickle.load(handle)

In [3]:
# Load LFP data
start_time = 0.0
end_time = 0.5
padding = 0.5
V1 = Allen_dataset(fps=1000,
                   start_time=start_time, 
                   end_time=end_time,
                   padding=padding,
#                    orientation=[0],
                   session_id=757216464,
                   selected_probes=['probeA', 'probeB', 'probeC', 'probeD', 'probeE', 'probeF'],
#                    temporal_frequency=[1,2,4],
                   stimulus_condition_id=[275, 277, 246, 255, 272, 248, 283, 266, 274, 276, 286, 271, 268, 270],
                   stimulus_name='drifting_gratings')

# V1.get_lfp()
# V1.remove_padding(padding)
V1.get_trial_metric_per_unit_per_trial()
V1.get_running(method="mine")

/home/qix/anaconda3/lib/python3.9/site-packages/allensdk/brain_observatory/ecephys/stimulus_table/naming_utilities.py:154: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  movie_rows = table[stim_colname].str.contains(movie_re, na=False)
/home/qix/anaconda3/lib/python3.9/site-packages/allensdk/brain_observatory/ecephys/ecephys_session.py:1315: UserWarning: Session includes invalid time intervals that could be accessed with the attribute 'invalid_times',Spikes within these intervals are invalid and may need to be excluded from the analysis.
  warnings.warn("Session includes invalid time intervals that could be accessed with the attribute 'invalid_times',"


# Membership

In [6]:
membership[0]

,probe,group_id
unit_ids,,
951814834,probeA,2
951814827,probeA,1
951814874,probeA,2
951814839,probeA,1
951814898,probeA,1
...,...,...
951804296,probeF,2
951804325,probeF,2
951804346,probeF,2


In [19]:
# Find total number of neurons needed and their corresponding trials where they are classified as cross-pop
target_probe = 'probeD'
cross_pop_list = []
cross_pop_conditions = {}
for neuron in membership[0].index:
    if membership[1].loc[neuron]['probe'] == target_probe:
        for i, member in enumerate(membership):
            if member.loc[neuron]['group_id'] == 0:
                if neuron not in cross_pop_conditions:
                    cross_pop_list.append(neuron)
                    cross_pop_conditions[neuron] = [condition_ids[i]]
                else:
                    cross_pop_conditions[neuron].append(condition_ids[i])
major_cross_pop_list = []
for neuron in cross_pop_list:
    if len(cross_pop_conditions[neuron])>= 40:
        major_cross_pop_list.append(neuron)
n_neuron = len(major_cross_pop_list)

In [20]:
# Neurons that are classified as cross-pop in at least one conditions.
cross_pop_list

[951810863,
 951810983,
 951811026,
 951811015,
 951812383,
 951811152,
 951811141,
 951812394,
 951811203,
 951811292,
 951811282,
 951811302,
 951812405,
 951811389,
 951811428,
 951811497,
 951811674,
 951811873,
 951812614]

In [21]:
# Neurons that 
major_cross_pop_list

[951810983, 951811026, 951811015]

# Fit a single neuron for example

In [57]:
spike_train_ind = np.zeros((V1.nt+V1.npadding, V1.spike_train.shape[1]))
for itrial in range(V1.spike_train.shape[1]):
    spike_train_ind[:,itrial] = V1.spike_train.loc[951811026, V1.spike_train.columns[itrial]]
    
coupling_filter_params = {'peaks_max':50, 'num':6, 'nonlinear':0.3}
pillow_basis = GLM.make_pillow_basis(**coupling_filter_params)
X_history = GLM.conv(spike_train_ind, pillow_basis, npadding=V1.npadding)

In [58]:
spike_train_ind.shape, X_history.shape

((1000, 210), (105000, 6))

In [43]:
# Get time shifts; peak1 peak2 timing
# The following hyperparameters turned out to be the best

max_iter = 10
self_effect = False
target_probe = 'probeD'
tau = 10
coupling_filter_params = {'peaks_max':50, 'num':6, 'nonlinear':0.3}
coupling_filter_params1 = {'peaks_max':20, 'num':2, 'nonlinear':0.3}
coupling_filter_params2 = {'peaks_max':50, 'num':5, 'nonlinear':0.3}
num_basis_baseline = 30
penalty = 3e-1

########## Nothing need to be changed below
probe_list = V1.selected_probes

select_trials = V1.stationary_trial_index
model_stationary = GLM.PP_GLM(dataset=V1, 
                   select_trials=select_trials, 
                   membership=membership, 
                   condition_ids=condition_ids)
model_stationary.add_effect('inhomogeneous_baseline', num=num_basis_baseline, apply_no_penalty=True)
for j, input_probe in enumerate(probe_list):
    model_stationary.add_effect('coupling', probe_list[j], **coupling_filter_params, apply_no_penalty=True)
model_stationary.add_effect('trial_coef')
# model_stationary.fit_time_warping_baseline(spike_train_ind[:,select_trials], method='mine', verbose=False, 
#                                            max_iter=max_iter, penalty=penalty)
model_stationary.fit_time_warping_baseline(target_probe, method='mine', verbose=False, 
                                           max_iter=max_iter, penalty=penalty)

select_trials = V1.running_trial_index
model_running = GLM.PP_GLM(dataset=V1, 
                   select_trials=select_trials, 
                   membership=membership, 
                   condition_ids=condition_ids)
model_running.add_effect('inhomogeneous_baseline', num=num_basis_baseline, apply_no_penalty=True)
for j, input_probe in enumerate(probe_list):
    model_running.add_effect('coupling', probe_list[j], **coupling_filter_params, apply_no_penalty=True)
model_running.add_effect('trial_coef')
model_running.fit_time_warping_baseline(target_probe, method='mine', verbose=False, 
                                        max_iter=max_iter, penalty=penalty)

fix_shifts_stationary = model_stationary.shifts
fix_shifts_running = model_running.shifts

In [59]:
# The following hyperparameters turned out to be the best

max_iter = 10
self_effect = False
target_probe = 'probeC'
tau = 10
coupling_filter_params = {'peaks_max':50, 'num':6, 'nonlinear':0.3}
coupling_filter_params1 = {'peaks_max':20, 'num':2, 'nonlinear':0.3}
coupling_filter_params2 = {'peaks_max':50, 'num':5, 'nonlinear':0.3}
num_basis_baseline = 30
penalty = 3e-1

########## Nothing need to be changed below
probe_list = V1.selected_probes

select_trials = V1.stationary_trial_index
model_stationary = GLM.PP_GLM(dataset=V1, 
                   select_trials=select_trials, 
                   membership=membership, 
                   condition_ids=condition_ids)
model_stationary.add_effect('inhomogeneous_baseline', num=num_basis_baseline, apply_no_penalty=True)
for j, input_probe in enumerate(probe_list):
    model_stationary.add_effect('coupling', probe_list[j], **coupling_filter_params, apply_no_penalty=True)
model_stationary.add_effect('trial_coef')
model_stationary.fit_time_warping_baseline(spike_train_ind[:,select_trials], method='mine', verbose=False, 
                                           max_iter=max_iter, penalty=penalty, fix_shifts=fix_shifts_stationary)

select_trials = V1.running_trial_index
model_running = GLM.PP_GLM(dataset=V1, 
                   select_trials=select_trials, 
                   membership=membership, 
                   condition_ids=condition_ids)
model_running.add_effect('inhomogeneous_baseline', num=num_basis_baseline, apply_no_penalty=True)
for j, input_probe in enumerate(probe_list):
    model_running.add_effect('coupling', probe_list[j], **coupling_filter_params, apply_no_penalty=True)
model_running.add_effect('trial_coef')
model_running.fit_time_warping_baseline(spike_train_ind[:,select_trials], method='mine', verbose=False, 
                                        max_iter=max_iter, penalty=penalty, fix_shifts=fix_shifts_running)

In [60]:
model_stationary.aic, model_running.aic

(5579.250682040347, 6771.221971766246)

In [61]:
# The following hyperparameters turned out to be the best

max_iter = 10
self_effect = False
target_probe = 'probeC'
tau = 10
coupling_filter_params = {'peaks_max':50, 'num':6, 'nonlinear':0.3}
coupling_filter_params1 = {'peaks_max':20, 'num':2, 'nonlinear':0.3}
coupling_filter_params2 = {'peaks_max':50, 'num':5, 'nonlinear':0.3}
num_basis_baseline = 30
penalty = 3e-1

########## Nothing need to be changed below
probe_list = V1.selected_probes

select_trials = V1.stationary_trial_index
model_stationary = GLM.PP_GLM(dataset=V1, 
                   select_trials=select_trials, 
                   membership=membership, 
                   condition_ids=condition_ids)
model_stationary.add_effect('inhomogeneous_baseline', num=num_basis_baseline, apply_no_penalty=True)
for j, input_probe in enumerate(probe_list):
    model_stationary.add_effect('coupling', probe_list[j], **coupling_filter_params, apply_no_penalty=True)
model_stationary.add_effect('trial_coef')
model_stationary.fit(spike_train_ind[:,select_trials], method='mine', verbose=False, 
                                           penalty=penalty)

select_trials = V1.running_trial_index
model_running = GLM.PP_GLM(dataset=V1, 
                   select_trials=select_trials, 
                   membership=membership, 
                   condition_ids=condition_ids)
model_running.add_effect('inhomogeneous_baseline', num=num_basis_baseline, apply_no_penalty=True)
for j, input_probe in enumerate(probe_list):
    model_running.add_effect('coupling', probe_list[j], **coupling_filter_params, apply_no_penalty=True)
model_running.add_effect('trial_coef')
model_running.fit(spike_train_ind[:,select_trials], method='mine', verbose=False, 
                                        penalty=penalty)

In [62]:
model_stationary.aic, model_running.aic

(5608.4410086410035, 6782.454920317823)